# Codigo final V2: DroneFace + ArcFace con graficos livianos

## 0. Especificacion de diseno y protocolo

Este notebook mantiene un unico flujo final, pero separa dos modos de trabajo:

- **Pipeline pesado**: construye galerias, inicializa ArcFace/InsightFace, calcula embeddings, detecta rostros y evalua.
- **Analisis liviano**: lee CSV ya guardados en `outputs_v2/`, regenera tablas/graficos y muestra resultados.

Por defecto, el notebook queda en modo liviano:

```python
RUN_HEAVY_PIPELINE = False
REGENERATE_FIGURES = True
RUN_VERIFICATION_SCORING = False
```

Con `RUN_HEAVY_PIPELINE = False`, el notebook no inicializa InsightFace y no recalcula embeddings ni detecciones.
Con `RUN_VERIFICATION_SCORING = False`, el notebook tampoco recalcula scores verification; solo los lee si ya existen.

### Decision metodologica principal

No se ajustan los pesos de ArcFace. InsightFace/ArcFace se usa como modelo preentrenado (`buffalo_l`). Cuando se habla de "entrenar", significa construir galerias de embeddings de referencia.

### Galerias V2

Las galerias se construyen desde `trio_cam + trio_gp`, porque cubren los 11 sujetos (`a` a `k`) y tres poses por sujeto:

```text
trio_front = porF
trio_side  = porL + porR
trio_all   = porF + porL + porR
```

`photos_all_faces` ya esta etiquetado por nombre de archivo. `photos_all` usa la anotacion manual de orden espacial en `outputs_v2/annotations/photos_all_spatial_order_manual.csv`.

### Repeticiones e incertidumbre

No se reporta media/desviacion estandar de multiples corridas, porque el pipeline actual es deterministico: mismo modelo, mismos datos y mismas galerias producen los mismos resultados. Se reportan metricas puntuales: accuracy, precision, recall, F1, TP, FP y FN.

Para comparacion con el paper base se agrega un protocolo tipo verification/ROC: se calculan scores continuos entre cada probe y cada identidad candidata. La clase `None/unknown` se usa solo despues, al aplicar un umbral de decision; no forma parte del calculo de AUC.

Los heatmaps comparables con la figura del paper usan bins de distancia de 1.5 m:

```text
2.5, 4, 5.5, 7, 8.5, 10, 11.5, 13, 14.5, 16 m
```

El umbral de `None/unknown` se elige desde `photos_all_faces` con criterio ROC por pares (`Youden J = TPR - FPR`) y luego se aplica sin reajustar a `photos_all`.

Para `photos_all` se muestran dos ROC:

- **End-to-end**: incluye fallas de deteccion como score 0.
- **Detected-only**: excluye fallas de deteccion y evalua el reconocimiento solo cuando existe una deteccion.

Todos los resultados nuevos se guardan en `outputs_v2/`. La carpeta `outputs/` queda intacta.

## 1. Librerias, flags y configuracion

In [ ]:
from pathlib import Path

from IPython.display import Image as NotebookImage, display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.config import DRONEFACE_ROOT, PROJECT_ROOT
from src.v2_figures import (
    GALLERY_NAMES,
    OUTPUTS_V2_DIR,
    V2_FIGS_DIR,
    V2_RESULTS_DIR,
    V2_TABLES_DIR,
    bbox_top_level_metrics,
    load_v2_light_results,
    regenerate_v2_figures,
)
from src.v2_verification import (
    run_saved_verification_analysis,
    verification_score_paths,
)
from src.v2_bbox_size_analysis import run_bbox_size_analysis

pd.set_option("display.max_columns", 140)
pd.set_option("display.width", 180)

RUN_HEAVY_PIPELINE = False
REGENERATE_FIGURES = True
RUN_VERIFICATION_SCORING = False
RECOMPUTE_VERIFICATION_SCORES = False
VERIFICATION_GALLERY = "trio_front"

FACE_DET_SIZE = (320, 320)
FULL_IMAGE_DET_SIZE = (1280, 1280)

print(f"Project root: {PROJECT_ROOT}")
print(f"DroneFace root: {DRONEFACE_ROOT}")
print(f"V2 outputs: {OUTPUTS_V2_DIR}")
print(f"RUN_HEAVY_PIPELINE={RUN_HEAVY_PIPELINE}")
print(f"REGENERATE_FIGURES={REGENERATE_FIGURES}")
print(f"RUN_VERIFICATION_SCORING={RUN_VERIFICATION_SCORING}")
print(f"VERIFICATION_GALLERY={VERIFICATION_GALLERY}")

## 2. Indice de DroneFace

In [ ]:
drone_index_v2_path = V2_TABLES_DIR / "droneface_index_v2.csv"

if RUN_HEAVY_PIPELINE:
    from src.dataset_index import inspect_droneface_dataset, build_droneface_index
    from src.v2_protocol import select_photos_all_faces_probes

    drone_info = inspect_droneface_dataset(DRONEFACE_ROOT, n_examples=12)
    drone_df = build_droneface_index(DRONEFACE_ROOT, save=False)
    drone_index_v2_path.parent.mkdir(parents=True, exist_ok=True)
    drone_df.to_csv(drone_index_v2_path, index=False)
    print(f"Indice V2 guardado en: {drone_index_v2_path}")

    trio_counts = (
        drone_df[drone_df["source_folder"].isin(["trio_cam", "trio_gp"])]
        .groupby(["source_folder", "subject_id", "image_type"], dropna=False)
        .size()
        .reset_index(name="n")
        .sort_values(["source_folder", "subject_id", "image_type"])
    )
    for source in ["trio_cam", "trio_gp"]:
        subset = trio_counts[trio_counts["source_folder"] == source]
        assert len(subset) == 33, f"{source}: se esperaban 11 sujetos x 3 poses"
        assert subset["subject_id"].nunique() == 11, f"{source}: se esperaban 11 sujetos"
        assert sorted(subset["image_type"].unique()) == ["porF", "porL", "porR"]
        assert (subset["n"] == 1).all(), f"{source}: alguna combinacion sujeto-pose no tiene 1 imagen"
    print("Validacion trio_cam/trio_gp OK: 11 sujetos x 3 poses en cada fuente.")
    print(f"photos_all_faces probes disponibles: {len(select_photos_all_faces_probes(drone_df))}")
else:
    if drone_index_v2_path.exists():
        drone_df = pd.read_csv(drone_index_v2_path)
        print(f"Indice V2 cargado desde: {drone_index_v2_path}")
    else:
        drone_df = pd.DataFrame()
        print("No existe droneface_index_v2.csv. Para reconstruirlo, usa RUN_HEAVY_PIPELINE=True.")

if not drone_df.empty:
    display(
        drone_df.groupby(["source_folder", "split"], dropna=False)
        .size()
        .reset_index(name="n")
        .sort_values(["source_folder", "split"])
    )

## 3. Pipeline pesado opcional: Experimento A V2

Esta seccion solo corre si `RUN_HEAVY_PIPELINE=True`. Construye las galerias `trio_front`, `trio_side` y `trio_all`, y evalua `photos_all_faces`.

In [ ]:
gallery_outputs = None
pose_comparison_df = None
gallery_info_df = None

if RUN_HEAVY_PIPELINE:
    from src.face_model import FaceRecognizerArcFace
    from src.v2_protocol import run_pose_gallery_experiments

    model_faces = FaceRecognizerArcFace(model_name="buffalo_l", det_size=FACE_DET_SIZE, allow_cuda=False)
    print(f"Modelo A inicializado con det_size={model_faces.det_size}")

    gallery_outputs, pose_comparison_df, gallery_info_df = run_pose_gallery_experiments(
        drone_df,
        model_faces,
    )
    display(pose_comparison_df)
    display(gallery_info_df.head(20))
else:
    print("Saltado: RUN_HEAVY_PIPELINE=False. Se usaran los CSV de outputs_v2 en las secciones livianas.")

## 4. Pipeline pesado opcional: Experimento B V2

Esta seccion solo corre si `RUN_HEAVY_PIPELINE=True`. Evalua `photos_all` completo con las galerias construidas en el Experimento A.

In [ ]:
full_outputs = None
full_set_summary_df = None

if RUN_HEAVY_PIPELINE:
    from src.face_model import FaceRecognizerArcFace
    from src.v2_protocol import run_full_image_set_experiments

    if gallery_outputs is None:
        raise RuntimeError("Primero debe ejecutarse el Experimento A pesado para construir gallery_outputs.")

    model_full = FaceRecognizerArcFace(model_name="buffalo_l", det_size=FULL_IMAGE_DET_SIZE, allow_cuda=False)
    print(f"Modelo B inicializado con det_size={model_full.det_size}")

    full_outputs, full_set_summary_df = run_full_image_set_experiments(
        drone_df,
        model_full,
        gallery_outputs,
    )
    display(full_set_summary_df)
else:
    print("Saltado: RUN_HEAVY_PIPELINE=False.")

## 5. Pipeline pesado opcional: bbox-level con anotacion manual

Esta seccion solo corre si `RUN_HEAVY_PIPELINE=True`. Usa el orden espacial manual de `photos_all` y calcula TP, FP, FN, precision, recall y F1 por bounding box.

In [ ]:
bbox_ready = False
labeled_bbox_df = None
bbox_metrics_df = None

if RUN_HEAVY_PIPELINE:
    from src.v2_protocol import (
        V2_ANNOTATIONS_DIR,
        create_spatial_annotation_template,
        run_bbox_level_if_annotations_complete,
        validate_spatial_annotation,
    )

    annotation_path = create_spatial_annotation_template(
        V2_ANNOTATIONS_DIR / "photos_all_spatial_order_manual.csv",
        overwrite=False,
    )
    annotation_df, annotation_complete, annotation_errors = validate_spatial_annotation(annotation_path)
    print(f"Anotacion manual: {annotation_path}")
    print(f"Completa: {annotation_complete}")
    display(annotation_df)

    if full_outputs is None:
        raise RuntimeError("Primero debe ejecutarse el Experimento B pesado para construir full_outputs.")

    bbox_ready, labeled_bbox_df, bbox_metrics_df, bbox_errors = run_bbox_level_if_annotations_complete(
        drone_df,
        full_outputs,
        annotation_path=annotation_path,
    )
    if not bbox_ready:
        print("BBox-level omitido: la anotacion manual aun no esta completa.")
        for error in bbox_errors:
            print(f"- {error}")
    else:
        display(bbox_metrics_df)
else:
    print("Saltado: RUN_HEAVY_PIPELINE=False.")

## 6. Resultados livianos desde `outputs_v2`

Desde aqui no se inicializa InsightFace. Esta parte carga resultados ya calculados y permite ajustar tablas/graficos rapidamente.

In [ ]:
light_results = load_v2_light_results()

pose_comparison_df = light_results.pose_comparison
full_set_summary_df = light_results.full_set_summary
bbox_metrics_df = light_results.bbox_metrics
labeled_bbox_df = light_results.labeled_bbox

print("Experimento A: photos_all_faces")
display(
    pose_comparison_df[
        [
            "gallery_name",
            "poses",
            "gallery_rows",
            "gallery_subjects_with_embedding",
            "total_probe",
            "detected_count",
            "detection_rate",
            "correct_count",
            "accuracy_global",
            "accuracy_conditional",
        ]
    ]
)

print("Experimento B: photos_all por conjunto de identidades")
display(
    full_set_summary_df[
        [
            "gallery_name",
            "total_images",
            "images_with_detections",
            "image_detection_rate",
            "expected_identity_recall_global",
            "exact_expected_set_match_rate",
            "mean_jaccard_set_similarity",
        ]
    ]
)

print("Experimento B bbox-level")
bbox_top_df = bbox_top_level_metrics(bbox_metrics_df)
display(bbox_top_df[["gallery_name", "total_slots", "tp", "fp", "fn", "precision", "recall", "f1"]])

## 7. Regeneracion liviana de graficos

In [ ]:
if REGENERATE_FIGURES:
    figure_paths = regenerate_v2_figures(light_results)
    print(f"Graficos regenerados: {len(figure_paths)}")
else:
    figure_paths = sorted((V2_FIGS_DIR / "metrics").glob("*.png"))
    print(f"Graficos existentes encontrados: {len(figure_paths)}")

display(pd.DataFrame({"figure_path": [str(path) for path in figure_paths]}))

## 8. Analisis de tamano de rostro detectado

Esta seccion usa solo `photos_all_detections_labeled_all_galleries.csv`; no recalcula detecciones ni embeddings. El objetivo es medir si la confiabilidad del reconocimiento cae cuando la caja del rostro es demasiado pequena.

In [ ]:
bbox_size_analysis = run_bbox_size_analysis(VERIFICATION_GALLERY)
bbox_size_df = bbox_size_analysis.detections
bbox_size_metrics_df = bbox_size_analysis.metrics

display(bbox_size_metrics_df)
print("Figuras bbox-size:")
for fig_path in bbox_size_analysis.figure_paths:
    print(fig_path)

## 9. Visualizaciones principales

In [ ]:
main_figures = [
    V2_FIGS_DIR / "metrics" / "photos_all_faces_pose_comparison.png",
    V2_FIGS_DIR / "metrics" / "photos_all_bbox_f1_by_gallery.png",
    V2_FIGS_DIR / "metrics" / "photos_all_bbox_precision_recall_f1_by_gallery.png",
    V2_FIGS_DIR / "metrics" / "photos_all_bbox_tp_fp_fn_by_gallery.png",
    V2_FIGS_DIR / "metrics" / "photos_all_bbox_f1_by_prefix.png",
    V2_FIGS_DIR / "metrics" / "photos_all_bbox_f1_by_height.png",
    V2_FIGS_DIR / "metrics" / "photos_all_bbox_f1_by_distance.png",
    V2_FIGS_DIR / "metrics" / "photos_all_bbox_accuracy_by_bbox_height.png",
    V2_FIGS_DIR / "metrics" / "photos_all_bbox_size_vs_distance.png",
    V2_FIGS_DIR / "detection" / f"photos_all_bbox_zoom_examples_{VERIFICATION_GALLERY}.png",
]

for fig_path in main_figures:
    if fig_path.exists():
        print(fig_path.name)
        display(NotebookImage(filename=str(fig_path)))
    else:
        print(f"No existe: {fig_path}")

## 10. Verification/AUC comparable con el paper

Esta seccion permite comparar contra el enfoque tipo verification del paper. El AUC se calcula desde pares `(probe, identidad candidata)`.

`None/unknown` no se usa para calcular AUC; aparece solo al aplicar un umbral. El umbral se elige desde `photos_all_faces` con el criterio ROC por pares `Youden J = TPR - FPR` y se aplica igual a `photos_all`.

La ROC end-to-end incluye las fallas de deteccion como score 0. La ROC detected-only excluye esas filas y permite mirar reconocimiento condicionado a que el detector encuentre la cara.

```text
if best_similarity < threshold:
    predicted_subject = None
else:
    predicted_subject = best_subject
```

In [ ]:
score_paths = verification_score_paths(VERIFICATION_GALLERY)

if RUN_VERIFICATION_SCORING:
    from src.face_model import FaceRecognizerArcFace
    from src.v2_verification import (
        build_photos_all_bbox_verification_scores,
        build_photos_all_faces_verification_scores,
        load_gallery_embeddings,
    )

    gallery_embeddings = load_gallery_embeddings(VERIFICATION_GALLERY)

    if RECOMPUTE_VERIFICATION_SCORES or not score_paths.faces_scores.exists():
        faces_results_df = pd.read_csv(V2_RESULTS_DIR / f"photos_all_faces_{VERIFICATION_GALLERY}.csv")
        model_faces = FaceRecognizerArcFace(model_name="buffalo_l", det_size=FACE_DET_SIZE, allow_cuda=False)
        build_photos_all_faces_verification_scores(
            faces_results_df,
            model_faces,
            gallery_embeddings,
            gallery_name=VERIFICATION_GALLERY,
            save_path=score_paths.faces_scores,
        )
        print(f"Scores photos_all_faces guardados en: {score_paths.faces_scores}")
    else:
        print(f"Scores photos_all_faces existentes: {score_paths.faces_scores}")

    if RECOMPUTE_VERIFICATION_SCORES or not score_paths.bbox_scores.exists():
        labeled_bbox_df = pd.read_csv(V2_RESULTS_DIR / "photos_all_detections_labeled_all_galleries.csv")
        model_full = FaceRecognizerArcFace(model_name="buffalo_l", det_size=FULL_IMAGE_DET_SIZE, allow_cuda=False)
        build_photos_all_bbox_verification_scores(
            labeled_bbox_df,
            model_full,
            gallery_embeddings,
            gallery_name=VERIFICATION_GALLERY,
            save_path=score_paths.bbox_scores,
        )
        print(f"Scores bbox guardados en: {score_paths.bbox_scores}")
    else:
        print(f"Scores bbox existentes: {score_paths.bbox_scores}")
else:
    print("Scoring verification omitido: RUN_VERIFICATION_SCORING=False.")
    print(f"CSV esperado photos_all_faces: {score_paths.faces_scores}")
    print(f"CSV esperado bbox: {score_paths.bbox_scores}")

In [ ]:
if score_paths.faces_scores.exists() and score_paths.bbox_scores.exists():
    verification_analysis = run_saved_verification_analysis(VERIFICATION_GALLERY)
    verification_auc_summary_df = verification_analysis["auc_summary"]
    verification_threshold_summary_df = verification_analysis["threshold_summary"]
    verification_threshold = verification_analysis["threshold"]

    print(f"Threshold verification elegido desde photos_all_faces: {verification_threshold:.6f}")
    print("AUC global:")
    display(verification_auc_summary_df)
    print("Metricas con None/unknown por umbral:")
    display(verification_threshold_summary_df)
else:
    verification_analysis = None
    verification_auc_summary_df = pd.DataFrame()
    verification_threshold_summary_df = pd.DataFrame()
    print("No hay CSV de scores verification. Activa RUN_VERIFICATION_SCORING=True para generarlos.")

In [ ]:
verification_figures = [
    V2_FIGS_DIR / "metrics" / f"photos_all_faces_verification_{VERIFICATION_GALLERY}_roc.png",
    V2_FIGS_DIR / "metrics" / f"photos_all_faces_verification_{VERIFICATION_GALLERY}_roc_detected_only.png",
    V2_FIGS_DIR / "metrics" / f"photos_all_faces_verification_{VERIFICATION_GALLERY}_score_histogram.png",
    V2_FIGS_DIR / "metrics" / f"photos_all_faces_verification_{VERIFICATION_GALLERY}_score_histogram_detected_only.png",
    V2_FIGS_DIR / "metrics" / f"photos_all_faces_verification_{VERIFICATION_GALLERY}_f1_vs_threshold.png",
    V2_FIGS_DIR / "metrics" / f"photos_all_faces_verification_{VERIFICATION_GALLERY}_decision_f1_vs_threshold.png",
    V2_FIGS_DIR / "metrics" / f"photos_all_faces_verification_{VERIFICATION_GALLERY}_auc_paper_height_distance.png",
    V2_FIGS_DIR / "metrics" / f"photos_all_faces_verification_{VERIFICATION_GALLERY}_auc_height_distance.png",
    V2_FIGS_DIR / "metrics" / f"photos_all_bbox_verification_{VERIFICATION_GALLERY}_roc.png",
    V2_FIGS_DIR / "metrics" / f"photos_all_bbox_verification_{VERIFICATION_GALLERY}_roc_detected_only.png",
    V2_FIGS_DIR / "metrics" / f"photos_all_bbox_verification_{VERIFICATION_GALLERY}_score_histogram.png",
    V2_FIGS_DIR / "metrics" / f"photos_all_bbox_verification_{VERIFICATION_GALLERY}_score_histogram_detected_only.png",
    V2_FIGS_DIR / "metrics" / f"photos_all_bbox_verification_{VERIFICATION_GALLERY}_f1_vs_threshold.png",
    V2_FIGS_DIR / "metrics" / f"photos_all_bbox_verification_{VERIFICATION_GALLERY}_decision_f1_vs_threshold.png",
    V2_FIGS_DIR / "metrics" / f"photos_all_bbox_verification_{VERIFICATION_GALLERY}_auc_paper_height_distance.png",
    V2_FIGS_DIR / "metrics" / f"photos_all_bbox_verification_{VERIFICATION_GALLERY}_auc_height_distance.png",
]

for fig_path in verification_figures:
    if fig_path.exists():
        print(fig_path.name)
        display(NotebookImage(filename=str(fig_path)))
    else:
        print(f"No existe: {fig_path}")

## 11. Discusion y conclusiones

In [ ]:
discussion_lines = []
discussion_lines.append("La version V2 usa galerias desde trio_cam + trio_gp y no hace fine-tuning de ArcFace.")
discussion_lines.append("El modo liviano permite regenerar tablas y graficos sin inicializar InsightFace ni repetir embeddings/detecciones.")
discussion_lines.append("No se reportan multiples runs porque el pipeline es deterministico bajo la configuracion actual.")

if not pose_comparison_df.empty:
    best_a = pose_comparison_df.sort_values("accuracy_global", ascending=False).iloc[0]
    discussion_lines.append(
        f"En photos_all_faces, la mejor galeria por accuracy_global fue {best_a['gallery_name']} "
        f"con {best_a['accuracy_global']:.2%}."
    )

if not full_set_summary_df.empty:
    best_b = full_set_summary_df.sort_values("expected_identity_recall_global", ascending=False).iloc[0]
    discussion_lines.append(
        f"En photos_all por conjunto, la mejor galeria por recall global fue {best_b['gallery_name']} "
        f"con {best_b['expected_identity_recall_global']:.2%}."
    )

if not bbox_top_df.empty:
    best_bbox = bbox_top_df.sort_values("f1", ascending=False).iloc[0]
    discussion_lines.append(
        f"En bbox-level, la mejor galeria por F1 fue {best_bbox['gallery_name']} "
        f"con F1={best_bbox['f1']:.2%}, precision={best_bbox['precision']:.2%}, recall={best_bbox['recall']:.2%}."
    )

if "verification_analysis" in globals() and verification_analysis is not None and not verification_auc_summary_df.empty:
    faces_auc = verification_auc_summary_df[
        (verification_auc_summary_df["protocol"].astype(str) == "photos_all_faces")
        & (verification_auc_summary_df["group"].astype(str) == "global")
    ]["auc"].iloc[0]
    faces_auc_detected = verification_auc_summary_df[
        (verification_auc_summary_df["protocol"].astype(str) == "photos_all_faces")
        & (verification_auc_summary_df["group"].astype(str) == "detected_only")
    ]["auc"].iloc[0]
    bbox_auc = verification_auc_summary_df[
        (verification_auc_summary_df["protocol"].astype(str) == "photos_all_bbox")
        & (verification_auc_summary_df["group"].astype(str) == "global")
    ]["auc"].iloc[0]
    bbox_auc_detected = verification_auc_summary_df[
        (verification_auc_summary_df["protocol"].astype(str) == "photos_all_bbox")
        & (verification_auc_summary_df["group"].astype(str) == "detected_only")
    ]["auc"].iloc[0]
    discussion_lines.append(
        f"En verification/AUC con {VERIFICATION_GALLERY}, photos_all_faces obtuvo AUC={faces_auc:.3f} "
        f"y photos_all bbox-level obtuvo AUC={bbox_auc:.3f}."
    )
    discussion_lines.append(
        f"Condicionado a detecciones, photos_all_faces obtuvo AUC={faces_auc_detected:.3f} "
        f"y photos_all bbox-level obtuvo AUC={bbox_auc_detected:.3f}."
    )
    discussion_lines.append(
        "La clase None/unknown se aplica solo al umbral de decision; AUC usa los scores continuos antes del umbral."
    )

if "bbox_size_metrics_df" in globals() and not bbox_size_metrics_df.empty:
    first_bin = bbox_size_metrics_df[bbox_size_metrics_df["bbox_height_bin"].astype(str) == "<30 px"]
    above_30 = bbox_size_metrics_df[bbox_size_metrics_df["bbox_height_bin"].astype(str) != "<30 px"]
    if not first_bin.empty and not above_30.empty:
        min_above_30 = above_30["accuracy_conditional"].dropna().min()
        discussion_lines.append(
            f"En el analisis condicionado a deteccion, las cajas bajo 30 px tuvieron "
            f"accuracy={first_bin['accuracy_conditional'].iloc[0]:.2%}; desde 30 px, "
            f"el peor bin quedo en {min_above_30:.2%}."
        )

for line in discussion_lines:
    print(f"- {line}")

summary_path = V2_TABLES_DIR / "v2_discussion_summary.txt"
summary_path.write_text("\n".join(discussion_lines), encoding="utf-8")
print(f"Resumen guardado en: {summary_path}")